# Step 01_02_03 -- Raw Schema DESCRIBE: aoe2companion

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoe2companion
**Question:** What are the column names and types for every raw data source in
aoe2companion? Since step 01_02_02 has not been executed there is no persistent
DuckDB yet; this notebook uses in-memory DuckDB + direct file reads with
`LIMIT 0` to obtain schema-only results without loading row data.
**Invariants applied:** #6 (reproducibility — SQL inlined in artifact), #9 (step scope: query)
**Step scope:** query
**Type:** Read-only — no DuckDB writes, no new tables, no schema changes

In [1]:
import json

import duckdb

from rts_predict.common.notebook_utils import get_reports_dir, setup_notebook_logging
from rts_predict.games.aoe2.config import (
    AOE2COMPANION_RAW_MATCHES_DIR,
    AOE2COMPANION_RAW_LEADERBOARDS_DIR,
    AOE2COMPANION_RAW_PROFILES_DIR,
    AOE2COMPANION_RAW_RATINGS_DIR,
)

logger = setup_notebook_logging()
logger.info("Matches dir: %s", AOE2COMPANION_RAW_MATCHES_DIR)
logger.info("Ratings dir: %s", AOE2COMPANION_RAW_RATINGS_DIR)
logger.info("Leaderboards dir: %s", AOE2COMPANION_RAW_LEADERBOARDS_DIR)
logger.info("Profiles dir: %s", AOE2COMPANION_RAW_PROFILES_DIR)

12:58:07 INFO notebook: Matches dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/raw/matches


12:58:07 INFO notebook: Ratings dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/raw/ratings


12:58:07 INFO notebook: Leaderboards dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/raw/leaderboards


12:58:07 INFO notebook: Profiles dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/raw/profiles


## 1. Connect to in-memory DuckDB

In [2]:
con = duckdb.connect(":memory:")
print("Connected to in-memory DuckDB")

Connected to in-memory DuckDB


## 2. DESCRIBE all raw sources

Sources covered:
- `matches`: glob `*.parquet` with `binary_as_string=True, union_by_name=True, filename=True`
- `ratings`: glob `*.csv` with explicit dtypes + `union_by_name=True, filename=True`
- `leaderboards`: single file `leaderboard.parquet` with `binary_as_string=True, filename=True`
- `profiles`: single file `profile.parquet` with `binary_as_string=True, filename=True`

`LIMIT 0` makes each query schema-only — no row data is loaded into memory.

In [3]:
schemas: dict[str, list[dict]] = {}

# --- matches ---
matches_glob = str(AOE2COMPANION_RAW_MATCHES_DIR / "*.parquet")
df_matches = con.execute("""
    DESCRIBE SELECT * FROM read_parquet(?, binary_as_string=true, union_by_name=true, filename=true) LIMIT 0
""", [matches_glob]).df()
schemas["matches"] = df_matches.to_dict(orient="records")
print(f"\n=== DESCRIBE matches ({len(df_matches)} columns) ===")
print(df_matches.to_string(index=False))


=== DESCRIBE matches (55 columns) ===
          column_name column_type null  key default extra
              matchId     INTEGER  YES None    None  None
              started   TIMESTAMP  YES None    None  None
             finished   TIMESTAMP  YES None    None  None
          leaderboard     VARCHAR  YES None    None  None
                 name     VARCHAR  YES None    None  None
               server     VARCHAR  YES None    None  None
internalLeaderboardId     INTEGER  YES None    None  None
              privacy     VARCHAR  YES None    None  None
                  mod     BOOLEAN  YES None    None  None
                  map     VARCHAR  YES None    None  None
           difficulty     VARCHAR  YES None    None  None
          startingAge     VARCHAR  YES None    None  None
         fullTechTree     BOOLEAN  YES None    None  None
          allowCheats     BOOLEAN  YES None    None  None
       empireWarsMode     BOOLEAN  YES None    None  None
            endingAge     VARCHAR

In [4]:
# --- ratings ---
ratings_glob = str(AOE2COMPANION_RAW_RATINGS_DIR / "*.csv")
df_ratings = con.execute("""
    DESCRIBE SELECT * FROM read_csv(
        ?,
        union_by_name=true,
        filename=true,
        dtypes={
            'profile_id': 'BIGINT',
            'games': 'BIGINT',
            'rating': 'BIGINT',
            'date': 'TIMESTAMP',
            'leaderboard_id': 'BIGINT',
            'rating_diff': 'BIGINT',
            'season': 'BIGINT'
        }
    ) LIMIT 0
""", [ratings_glob]).df()
schemas["ratings"] = df_ratings.to_dict(orient="records")
print(f"\n=== DESCRIBE ratings ({len(df_ratings)} columns) ===")
print(df_ratings.to_string(index=False))


=== DESCRIBE ratings (8 columns) ===
   column_name column_type null  key default extra
    profile_id      BIGINT  YES None    None  None
         games      BIGINT  YES None    None  None
        rating      BIGINT  YES None    None  None
          date   TIMESTAMP  YES None    None  None
leaderboard_id      BIGINT  YES None    None  None
   rating_diff      BIGINT  YES None    None  None
        season      BIGINT  YES None    None  None
      filename     VARCHAR  YES None    None  None


In [5]:
# --- leaderboards ---
leaderboard_file = str(AOE2COMPANION_RAW_LEADERBOARDS_DIR / "leaderboard.parquet")
df_leaderboards = con.execute("""
    DESCRIBE SELECT * FROM read_parquet(?, binary_as_string=true, filename=true) LIMIT 0
""", [leaderboard_file]).df()
schemas["leaderboards"] = df_leaderboards.to_dict(orient="records")
print(f"\n=== DESCRIBE leaderboards ({len(df_leaderboards)} columns) ===")
print(df_leaderboards.to_string(index=False))


=== DESCRIBE leaderboards (19 columns) ===
  column_name column_type null  key default extra
  leaderboard     VARCHAR  YES None    None  None
    profileId     INTEGER  YES None    None  None
         name     VARCHAR  YES None    None  None
         rank     INTEGER  YES None    None  None
       rating     INTEGER  YES None    None  None
lastMatchTime   TIMESTAMP  YES None    None  None
        drops     INTEGER  YES None    None  None
       losses     INTEGER  YES None    None  None
       streak     INTEGER  YES None    None  None
         wins     INTEGER  YES None    None  None
        games     INTEGER  YES None    None  None
    updatedAt   TIMESTAMP  YES None    None  None
  rankCountry     INTEGER  YES None    None  None
       active     BOOLEAN  YES None    None  None
       season     INTEGER  YES None    None  None
    rankLevel     INTEGER  YES None    None  None
      steamId     VARCHAR  YES None    None  None
      country     VARCHAR  YES None    None  None
     f

In [6]:
# --- profiles ---
profile_file = str(AOE2COMPANION_RAW_PROFILES_DIR / "profile.parquet")
df_profiles = con.execute("""
    DESCRIBE SELECT * FROM read_parquet(?, binary_as_string=true, filename=true) LIMIT 0
""", [profile_file]).df()
schemas["profiles"] = df_profiles.to_dict(orient="records")
print(f"\n=== DESCRIBE profiles ({len(df_profiles)} columns) ===")
print(df_profiles.to_string(index=False))


=== DESCRIBE profiles (14 columns) ===
       column_name column_type null  key default extra
         profileId     INTEGER  YES None    None  None
           steamId     VARCHAR  YES None    None  None
              name     VARCHAR  YES None    None  None
              clan     VARCHAR  YES None    None  None
           country     VARCHAR  YES None    None  None
        avatarhash     VARCHAR  YES None    None  None
     sharedHistory     BOOLEAN  YES None    None  None
     twitchChannel     VARCHAR  YES None    None  None
    youtubeChannel     VARCHAR  YES None    None  None
youtubeChannelName     VARCHAR  YES None    None  None
         discordId     VARCHAR  YES None    None  None
       discordName     VARCHAR  YES None    None  None
 discordInvitation     VARCHAR  YES None    None  None
          filename     VARCHAR  YES None    None  None


## 3. Write artifact

In [7]:
artifacts_dir = (
    get_reports_dir("aoe2", "aoe2companion")
    / "artifacts" / "01_exploration" / "02_eda"
)
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_data = {
    "step": "01_02_03",
    "dataset": "aoe2companion",
    "schemas": schemas,
}

artifact_path = artifacts_dir / "01_02_03_raw_schema_describe.json"
artifact_path.write_text(json.dumps(artifact_data, indent=2, default=str))
logger.info("Artifact written: %s", artifact_path)

print(f"\nArtifact written: {artifact_path}")
for source, cols in schemas.items():
    print(f"  {source}: {len(cols)} columns")

12:58:09 INFO notebook: Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json



Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json
  matches: 55 columns
  ratings: 8 columns
  leaderboards: 19 columns
  profiles: 14 columns


In [8]:
con.close()